# Test: choose an area in Jupyter (`duckosm.AreaPicker`)

This notebook tests the widget on Monaco. Run the cells in sequence.
You must have `pip install "duckosm[widget]"` and a connection to the internet.
The guide is `docs/guides/area-picker.md`.

## 1. Show the widget

Make sure that you see a map with the tabs Search, OSM id, Box, H3 cell and File.
Try each tab. For each choice, make sure that the line under the map shows the extract and its size,
and that the map shows the boundary in blue. Do not click **Get** yet.

In [ ]:
import time
import duckosm

picker = duckosm.AreaPicker(out="data")
picker

## 2. Set the area from Python (a box around Monaco)

The widget runs `duckosm get --dry-run`: it finds the boundary and the extract, and downloads nothing.
The map above moves to Monaco.

In [ ]:
picker.area = {"mode": "bbox", "bbox": [7.40, 43.72, 7.44, 43.76], "name": "monaco_box"}
for _ in range(300):                      # 60 s at most
    if picker.plan:
        break
    time.sleep(0.2)
assert "error" not in picker.plan, picker.plan
x = picker.plan["extract"]
print(picker.plan["name"], "->", x["name"], f"({x['id']}),", round(x["bytes"] / 2**20, 1), "MB,",
      "in the cache" if x["cached"] else "to download")
assert x["id"] == "monaco", x
assert picker.plan["boundary"]["geometry"]["type"] == "Polygon"

## 3. Get the OSM file

Click **Get** in the widget above, or run the next cell. The status line shows the progress.

In [ ]:
area = picker.get()
print(area)
assert area.pbf.is_file() and area.pbf.stat().st_size > 0
assert area.boundary.is_file()
assert picker.result is area

## 4. Wrong input gives an error, not a crash

The line under the map shows the error.

In [ ]:
picker.area = {"mode": "h3", "h3_cell": "not-a-cell"}
for _ in range(300):
    if picker.plan:
        break
    time.sleep(0.2)
print(picker.plan)
assert "error" in picker.plan

## 5. Search and a GeoJSON file

Do these steps in the widget:

1. In the tab **Search**, type `Monaco`, push Enter and click the match with the boundary. Make sure that the boundary of Monaco shows.
2. In the tab **File**, load a GeoJSON file from the computer of the browser. Step 3 wrote `data/monaco_box.geojson`: download it first from the JupyterLab file browser. Make sure that the box shows.
3. Click **Get** and make sure that the status line shows `wrote ...`.

In [ ]:
print(picker.area.get("mode"), picker.plan.get("extract", {}).get("id"), picker.status)